<a href="https://colab.research.google.com/github/Himanshu-bansal-9256/Himanshu_Bansal_JECRC_FOUNDATION_Celebal_Tech_Week_Assessments/blob/main/week7_%3CHimanshu_Bansal%3E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System using Retrieval-Augmented Generation (RAG)

This project implements a Retrieval-Augmented Generation (RAG) pipeline that answers questions from custom PDF and text documents.

Workflow:
1. Load Documents
2. Split into Chunks
3. Generate Embeddings
4. Store in Pinecone
5. Retrieve Relevant Chunks
6. Generate Answer using Gemini

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters pypdf google-generativeai pinecone python-dotenv

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import google.generativeai as genai
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# Load environment variables
load_dotenv()

# API Keys
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "rag-doc-qa-v2"

# Create Index if it doesn't exist
existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
    name=INDEX_NAME,
    dimension=3072,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index(INDEX_NAME)

print("=" * 60)
print("Gemini Connected")
print("Pinecone Connected")
print("=" * 60)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_32289/3920000343.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Gemini Connected
Pinecone Connected


In [ ]:
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=3072, total_vector_count=224, metric='cosine', namespaces=1)


In [ ]:
print("=" * 60)
print("Loading Documents...")
print("=" * 60)

docs = []
# Current directory
source_dir = Path(".")
for file in source_dir.iterdir():
    if file.suffix.lower() == ".pdf":
        print(f"Loading PDF : {file.name}")
        loader = PyPDFLoader(str(file))
        docs.extend(loader.load())
    elif file.suffix.lower() == ".txt":
        print(f"Loading TXT : {file.name}")
        loader = TextLoader(
            str(file),
            encoding="utf-8"
        )
        docs.extend(loader.load())

print("=" * 60)
print(f"Total Documents Loaded : {len(docs)}")
print("=" * 60)

for doc in docs[:3]:
    print(doc.metadata)

Loading Documents...
Loading PDF : dsa.pdf
Total Documents Loaded : 153
{'producer': 'www.ilovepdf.com', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2019-05-14T05:17:15+00:00', 'source': 'dsa.pdf', 'total_pages': 153, 'page': 0, 'page_label': '1'}
{'producer': 'www.ilovepdf.com', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2019-05-14T05:17:15+00:00', 'source': 'dsa.pdf', 'total_pages': 153, 'page': 1, 'page_label': '2'}
{'producer': 'www.ilovepdf.com', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2019-05-14T05:17:15+00:00', 'source': 'dsa.pdf', 'total_pages': 153, 'page': 2, 'page_label': '3'}


In [ ]:
print("=" * 60)
print("Splitting Documents into Chunks...")
print("=" * 60)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print(f"Total Chunks Created : {len(chunks)}")
print("=" * 60)

# Preview first chunk
print("\nFirst Chunk Preview:\n")
print(chunks[0].page_content[:500])

Splitting Documents into Chunks...
Total Chunks Created : 224

First Chunk Preview:

DATA STRUCTURES 
(R18A0503) 
 
 
LECTURE NOTES 
 
B.TECH II YEAR – I SEM (R18)  
(2019-20) 
 
 
 
 
 
 
 
 
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
(Recognized under 2(f) and 12 (B) of UGC ACT 1956) 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified)  
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telang


In [ ]:
def get_embedding(text):
    """
    Generate embedding for a document chunk using Gemini.
    """

    text = text.replace("\n", " ").strip()

    response = genai.embed_content(
    model="models/gemini-embedding-001",
    content=text,
    task_type="retrieval_document"
)

    return response["embedding"]

#### Get Query Embeddings

In [ ]:
def get_query_embedding(query):
    """
    Generate embedding for user query.
    """

    query = query.replace("\n", " ").strip()

    response = genai.embed_content(
    model="models/gemini-embedding-001",
    content=query,
    task_type="retrieval_query"
)

    return response["embedding"]

In [ ]:
from tqdm import tqdm

print("=" * 60)
print("Generating Embeddings...")
print("=" * 60)

vectors = []

for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):

    embedding = get_embedding(chunk.page_content)

    vectors.append({
        "id": str(i),
        "values": embedding,
        "metadata": {
            "text": chunk.page_content,
            "source": chunk.metadata.get("source", ""),
            "page": chunk.metadata.get("page", -1),
            "chunk_id": i
        }
    })

print(f"\nGenerated {len(vectors)} embeddings.")

print("=" * 60)
print("Uploading vectors to Pinecone...")
print("=" * 60)

batch_size = 5

for i in tqdm(range(0, len(vectors), batch_size)):

    batch = vectors[i:i + batch_size]

    index.upsert(vectors=batch)

print("=" * 60)
print("Upload Complete")
print("=" * 60)

print(index.describe_index_stats())

Generating Embeddings...


100%|██████████| 224/224 [04:26<00:00,  1.19s/it]



Generated 224 embeddings.
Uploading vectors to Pinecone...


100%|██████████| 45/45 [00:11<00:00,  3.95it/s]

Upload Complete
DescribeIndexStatsResponse(dimension=3072, total_vector_count=224, metric='cosine', namespaces=1)


In [ ]:
def search_docs(question, top_k=6):

    query_embedding = get_query_embedding(question)

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results["matches"]

In [ ]:
matches = search_docs("What is stack?")

print(f"Matches Found : {len(matches)}")

for i, match in enumerate(matches, start=1):

    print("="*60)

    print("Score :", round(match["score"],4))

    print("Source :", match["metadata"]["source"])

    print("Page :", match["metadata"]["page"])

    print(match["metadata"]["text"][:400])

Matches Found : 6
Score : 0.7438
Source : dsa.pdf
Page : 37
Page 1  
 
 
 
STACK ADT:- A Stack is a linear data structure  where insertion  and deletion of items takes place 
at one end called top of the stack. A Stack is defined as a data structure which operates on a last -in 
first-out basis. So it is also is referred as Last -in First-out( LIFO).  
Stack uses a single index or pointer to keep track of the information  in the stack. The basic 
operation
Score : 0.7119
Source : dsa.pdf
Page : 37
said to be popped off the stack. Two additional terms almost always used  with  stacks  are 
overflow, which occurs when we try to push more information on a stack that it can hold, and 
underflow, which occurs when we try to pop an item off a stack which is empty. 
 
 
Pushing items onto the stack:  
 
Assume that the array elements begin at 0 ( because the array subscript starts from 0)  
and th
Score : 0.6908
Source : dsa.pdf
Page : 37
template<class T> 
void stack<T>::push() 
{ 
if(top==m

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")

def ask(question):

    matches = search_docs(question)

    if len(matches)==0:
        print("No relevant documents found.")
        return

    context = "\n\n".join(
        match["metadata"]["text"]
        for match in matches
    )

    prompt = f"""
You are a helpful Document Question Answering assistant.

Read the entire context carefully before answering.

If the answer exists anywhere in the context, summarize it clearly in your own words.

If the answer is genuinely missing from the context, respond exactly with:

"I couldn't find the answer in the uploaded documents."

Do not use outside knowledge.
Do not make up information.
Always answer only from the retrieved context.

Context:
{context}

Question:
{question}

Answer:
"""

    response=model.generate_content(prompt)

    print("="*70)
    print("QUESTION")
    print("="*70)
    print(question)

    print("\n")

    print("="*70)
    print("ANSWER")
    print("="*70)
    print(response.text)

    print("\n")

    print("="*70)
    print("SOURCES")
    print("="*70)

    for match in matches:

        print(
            f"{match['metadata']['source']} | "
            f"Page {match['metadata']['page']} | "
            f"Score {match['score']:.4f}"
        )

In [ ]:
print("="*60)
print("Document Question Answering System")
print("="*60)

while True:

    question=input("\nAsk Question (type exit): ")

    if question.lower()=="exit":
        print("Good Bye")
        break

    ask(question)

Document Question Answering System

Ask Question (type exit): what is stack?
QUESTION
what is stack?


ANSWER
A stack is a linear data structure where items are inserted and deleted from only one end, known as the "top" of the stack. It operates on a last-in first-out (LIFO) basis, meaning the last item added is the first one to be removed. It uses a single index or pointer to keep track of information.


SOURCES
dsa.pdf | Page 37 | Score 0.7414
dsa.pdf | Page 37 | Score 0.7100
dsa.pdf | Page 37 | Score 0.6900
dsa.pdf | Page 42 | Score 0.6862
dsa.pdf | Page 45 | Score 0.6845
dsa.pdf | Page 43 | Score 0.6844

Ask Question (type exit): what is recusrion?
QUESTION
what is recusrion?


ANSWER
I couldn't find the answer in the uploaded documents.


SOURCES
dsa.pdf | Page 94 | Score 0.6744
dsa.pdf | Page 81 | Score 0.6586
dsa.pdf | Page 76 | Score 0.6568
dsa.pdf | Page 85 | Score 0.6559
dsa.pdf | Page 73 | Score 0.6531
dsa.pdf | Page 95 | Score 0.6456

Ask Question (type exit): what is linke

# Testing the System

After completing the implementation, I tested the RAG system with different questions from the uploaded document.

Some example questions include:

- What is Stack?
- What is Linked List?
- Explain Queue.
- Difference between Stack and Queue.

The system first retrieves the most relevant document chunks from Pinecone and then uses Gemini to generate an answer based only on the retrieved content.

The retrieved source document and similarity score are also displayed so that the answer can be verified.

# Conclusion

In this project, I built a simple Document Question Answering System using the RAG (Retrieval-Augmented Generation) approach.

The application loads PDF documents, splits them into smaller chunks, converts them into vector embeddings using the Gemini Embedding model, and stores them in Pinecone. When a user asks a question, the system retrieves the most relevant document chunks and generates an answer using Gemini 2.5 Flash.

This approach helps generate answers based on the uploaded documents instead of relying only on the language model's general knowledge.